# Royal Care Clinic - Healthcare Clinic Management System
**ICS220 Assignment 2** - Abdulla Alhammadi (202324776), Mohamed Salem (202115227)

### How this notebook works
The assignment asks us to put the classes in **separate Python files**. A Colab notebook is normally one file, so each code cell below starts with `%%writefile file_name.py`.

- `%%writefile person.py` saves the code in that cell as a file called `person.py` (it does not run it).
- The other files can then use it with `import`, for example `from person import Person`.

**To run:** click **Runtime > Run all**. The last two cells run the program (`main.py`) and the test cases (`test_clinic.py`). All 35 tests should show `ok`.

## 1. Person (parent class) - `person.py`

In [ ]:
%%writefile person.py
"""
person.py
Person is the parent class for Patient and Staff.
It stores the details that every person in the clinic has.
"""


class Person:
    """Parent class for everyone in the clinic (patients and staff)."""

    def __init__(self, person_id, name, phone, email):
        # Protected attributes (one underscore) so child classes can use them
        self._person_id = person_id
        self._name = name
        self._phone = phone
        self._email = email

    # ----- getters and setters -----
    def get_person_id(self):
        return self._person_id

    def get_name(self):
        return self._name

    def set_name(self, name):
        self._name = name

    def get_phone(self):
        return self._phone

    def set_phone(self, phone):
        self._phone = phone

    def get_email(self):
        return self._email

    def set_email(self, email):
        self._email = email

    def can_view_records(self):
        """By default a person cannot view other people's medical records.
        The Staff class overrides this method."""
        return False

    def __str__(self):
        return f"{self._name} (ID: {self._person_id})"


## 2. Appointment - `appointment.py`

In [ ]:
%%writefile appointment.py
"""
appointment.py
An Appointment is a 30-minute time slot in a doctor's schedule.
It starts as AVAILABLE, becomes BOOKED when a patient books it,
and COMPLETED when the doctor records the visit.
"""


class Appointment:
    """A time slot with a doctor that a patient can book."""

    # Possible statuses
    AVAILABLE = "AVAILABLE"
    BOOKED = "BOOKED"
    CANCELLED = "CANCELLED"
    COMPLETED = "COMPLETED"

    def __init__(self, appointment_id, doctor, date, time, duration=30):
        self.__appointment_id = appointment_id
        self.__doctor = doctor
        self.__date = date            # e.g. "2026-11-02"
        self.__time = time            # e.g. "09:30"
        self.__duration = duration    # minutes
        self.__patient = None         # no patient until booked
        self.__status = Appointment.AVAILABLE
        self.__record = None          # medical record after the visit

    # ----- getters -----
    def get_appointment_id(self):
        return self.__appointment_id

    def get_doctor(self):
        return self.__doctor

    def get_date(self):
        return self.__date

    def get_time(self):
        return self.__time

    def get_duration(self):
        return self.__duration

    def get_patient(self):
        return self.__patient

    def get_status(self):
        return self.__status

    def get_record(self):
        return self.__record

    def set_record(self, record):
        self.__record = record

    # ----- actions -----
    def is_available(self):
        """A slot can be booked if it is new or was cancelled."""
        return self.__status in (Appointment.AVAILABLE, Appointment.CANCELLED)

    def book(self, patient):
        """Book this slot for a patient."""
        if not self.is_available():
            raise ValueError("This appointment slot is already booked.")
        self.__patient = patient
        self.__status = Appointment.BOOKED

    def cancel(self):
        """Cancel the booking. The slot becomes free for other patients."""
        if self.__status != Appointment.BOOKED:
            raise ValueError("Only a booked appointment can be cancelled.")
        self.__patient = None
        self.__status = Appointment.CANCELLED

    def complete(self):
        """Mark the appointment as completed after the visit."""
        if self.__status != Appointment.BOOKED:
            raise ValueError("Only a booked appointment can be completed.")
        self.__status = Appointment.COMPLETED

    def __str__(self):
        patient = self.__patient.get_name() if self.__patient else "-"
        return (f"{self.__appointment_id} | {self.__date} {self.__time} "
                f"({self.__duration} min) | {self.__doctor.get_name()} | "
                f"Patient: {patient} | {self.__status}")


## 3. MedicalRecord, Prescription, LabOrder - `medical_record.py`

In [ ]:
%%writefile medical_record.py
"""
medical_record.py
Electronic Medical Record (EMR) for one visit, with its prescriptions
and lab orders.

MedicalRecord *-- Prescription  (composition)
MedicalRecord *-- LabOrder      (composition)
Prescriptions and lab orders are created by the medical record and
cannot exist without it.
"""


class Prescription:
    """A medicine prescribed during a visit."""

    CREATED = "CREATED"
    SENT_TO_PHARMACY = "SENT TO PHARMACY"

    def __init__(self, drug_name, dosage, directions, price):
        self.__drug_name = drug_name
        self.__dosage = dosage
        self.__directions = directions
        self.__price = price
        self.__status = Prescription.CREATED

    def get_drug_name(self):
        return self.__drug_name

    def get_dosage(self):
        return self.__dosage

    def get_directions(self):
        return self.__directions

    def get_price(self):
        return self.__price

    def get_status(self):
        return self.__status

    def send_to_pharmacy(self):
        """Send the prescription to the pharmacy."""
        self.__status = Prescription.SENT_TO_PHARMACY

    def __str__(self):
        return (f"{self.__drug_name} {self.__dosage} - {self.__directions} "
                f"[{self.__status}]")


class LabOrder:
    """A lab test ordered by the doctor. Status: ORDERED -> COMPLETED."""

    ORDERED = "ORDERED"
    COMPLETED = "COMPLETED"

    def __init__(self, test_name, price):
        self.__test_name = test_name
        self.__price = price
        self.__status = LabOrder.ORDERED
        self.__result = None     # filled in when the lab finishes

    def get_test_name(self):
        return self.__test_name

    def get_price(self):
        return self.__price

    def get_status(self):
        return self.__status

    def get_result(self):
        return self.__result

    def add_result(self, result):
        """Store the result. The order becomes COMPLETED."""
        if self.__status == LabOrder.COMPLETED:
            raise ValueError("This lab order already has a result.")
        self.__result = result
        self.__status = LabOrder.COMPLETED

    def __str__(self):
        result = self.__result if self.__result else "waiting"
        return f"{self.__test_name} [{self.__status}] Result: {result}"


class MedicalRecord:
    """The EMR of one visit: notes, diagnosis, vitals, prescriptions,
    lab orders and attachments."""

    def __init__(self, record_id, patient, doctor, appointment, diagnosis, notes, vitals):
        self.__record_id = record_id
        self.__patient = patient
        self.__doctor = doctor
        self.__appointment = appointment
        self.__diagnosis = diagnosis
        self.__notes = notes
        self.__vitals = vitals            # dictionary, e.g. {"BP": "120/80"}
        self.__prescriptions = []         # composition
        self.__lab_orders = []            # composition
        self.__attachments = []           # file names of scanned documents

    # ----- getters -----
    def get_record_id(self):
        return self.__record_id

    def get_patient(self):
        return self.__patient

    def get_doctor(self):
        return self.__doctor

    def get_appointment(self):
        return self.__appointment

    def get_diagnosis(self):
        return self.__diagnosis

    def get_notes(self):
        return self.__notes

    def get_vitals(self):
        return self.__vitals

    def get_prescriptions(self):
        return self.__prescriptions

    def get_lab_orders(self):
        return self.__lab_orders

    def get_attachments(self):
        return self.__attachments

    # ----- adding parts (composition: the record creates them) -----
    def add_prescription(self, drug_name, dosage, directions, price):
        prescription = Prescription(drug_name, dosage, directions, price)
        self.__prescriptions.append(prescription)
        return prescription

    def add_lab_order(self, test_name, price):
        lab_order = LabOrder(test_name, price)
        self.__lab_orders.append(lab_order)
        return lab_order

    def add_attachment(self, file_name):
        self.__attachments.append(file_name)

    # ----- access control -----
    def get_summary(self, viewer):
        """Return the visit summary. Only the patient or clinic staff may see it."""
        if viewer is not self.__patient and not viewer.can_view_records():
            raise PermissionError(f"{viewer.get_name()} cannot view this record.")

        lines = [f"Visit Summary - {self.__record_id}",
                 f"Patient   : {self.__patient.get_name()}",
                 f"Doctor    : {self.__doctor.get_name()}",
                 f"Date      : {self.__appointment.get_date()} {self.__appointment.get_time()}",
                 f"Diagnosis : {self.__diagnosis}",
                 f"Notes     : {self.__notes}",
                 f"Vitals    : {self.__vitals}",
                 "Prescriptions:"]
        for p in self.__prescriptions:
            lines.append(f"  - {p}")
        lines.append("Lab orders:")
        for lab in self.__lab_orders:
            lines.append(f"  - {lab}")
        lines.append(f"Attachments: {', '.join(self.__attachments) or 'None'}")
        return "\n".join(lines)


## 4. Staff and Doctor - `staff.py`

In [ ]:
%%writefile staff.py
"""
staff.py
Staff and Doctor classes.

Person <|-- Staff <|-- Doctor   (inheritance)
Doctor *-- Appointment          (composition: a doctor creates his own time slots)
"""

from person import Person
from appointment import Appointment
from medical_record import MedicalRecord


class Staff(Person):
    """Any clinic employee: nurse, receptionist, lab technician, doctor..."""

    def __init__(self, staff_id, name, phone, email, role, qualifications, working_hours):
        super().__init__(staff_id, name, phone, email)   # Person's attributes
        self._role = role                    # e.g. "Nurse"
        self._qualifications = qualifications
        self._working_hours = working_hours  # e.g. "Sun-Thu 08:00-16:00"
        self._department = None              # set when added to a department

    def get_role(self):
        return self._role

    def set_role(self, role):
        self._role = role

    def get_qualifications(self):
        return self._qualifications

    def get_working_hours(self):
        return self._working_hours

    def set_working_hours(self, working_hours):
        self._working_hours = working_hours

    def get_department(self):
        return self._department

    def set_department(self, department):
        self._department = department

    def can_view_records(self):
        """Overrides Person: staff members are allowed to view medical records."""
        return True

    def __str__(self):
        department = self._department.get_name() if self._department else "None"
        return f"{self._name} | {self._role} | {department} | {self._working_hours}"


class Doctor(Staff):
    """A doctor is a staff member with a speciality and a schedule."""

    record_counter = 0   # class attribute used to make record IDs

    def __init__(self, staff_id, name, phone, email, speciality, qualifications,
                 working_hours, consultation_fee):
        super().__init__(staff_id, name, phone, email, "Doctor", qualifications, working_hours)
        self.__speciality = speciality            # e.g. "General Practitioner"
        self.__consultation_fee = consultation_fee
        self.__appointments = []                  # composition: doctor's time slots

    def get_speciality(self):
        return self.__speciality

    def get_consultation_fee(self):
        return self.__consultation_fee

    def set_consultation_fee(self, fee):
        self.__consultation_fee = fee

    def get_appointments(self):
        return self.__appointments

    # ----- schedule -----
    def add_slot(self, date, time, duration=30):
        """Create a new available time slot in this doctor's schedule."""
        slot_id = f"{self._person_id}-{len(self.__appointments) + 1}"
        slot = Appointment(slot_id, self, date, time, duration)
        self.__appointments.append(slot)
        return slot

    def get_available_slots(self, date=None):
        """Free slots, optionally only on one date."""
        return [a for a in self.__appointments
                if a.is_available() and (date is None or a.get_date() == date)]

    # ----- clinical work -----
    def record_visit(self, appointment, diagnosis, notes, vitals):
        """Create the medical record for a booked appointment."""
        if appointment.get_doctor() is not self:
            raise PermissionError("This appointment belongs to another doctor.")
        Doctor.record_counter += 1
        record_id = f"EMR-{Doctor.record_counter:03d}"
        patient = appointment.get_patient()
        record = MedicalRecord(record_id, patient, self, appointment, diagnosis, notes, vitals)
        appointment.complete()            # raises an error if not booked
        appointment.set_record(record)
        patient.add_record(record)
        return record

    def prescribe(self, record, drug_name, dosage, directions, price):
        """Write a prescription and send it to the pharmacy."""
        prescription = record.add_prescription(drug_name, dosage, directions, price)
        prescription.send_to_pharmacy()
        return prescription

    def order_lab_test(self, record, test_name, price):
        return record.add_lab_order(test_name, price)

    def __str__(self):
        return super().__str__() + f" | {self.__speciality}"


## 5. Patient and Insurance - `patient.py`

In [ ]:
%%writefile patient.py
"""
patient.py
Patient and Insurance classes.

Person <|-- Patient     (inheritance)
Patient o-- Insurance   (aggregation: the insurance policy exists on its own,
                         the patient just has one)
"""

from person import Person


class Insurance:
    """A patient's insurance policy.
    The patient pays a fixed co-pay, then the insurance covers a
    percentage of the rest."""

    def __init__(self, provider, policy_number, coverage_percent, copay):
        self.__provider = provider
        self.__policy_number = policy_number
        self.__coverage_percent = coverage_percent   # e.g. 80
        self.__copay = copay                         # e.g. 50 (AED)

    def get_provider(self):
        return self.__provider

    def get_policy_number(self):
        return self.__policy_number

    def get_coverage_percent(self):
        return self.__coverage_percent

    def get_copay(self):
        return self.__copay

    def calculate_share(self, total):
        """Return (insurance pays, patient pays) for a bill total."""
        copay = min(self.__copay, total)
        insurance_pays = round((total - copay) * self.__coverage_percent / 100, 2)
        patient_pays = round(total - insurance_pays, 2)
        return insurance_pays, patient_pays

    def __str__(self):
        return (f"{self.__provider} ({self.__policy_number}) - "
                f"{self.__coverage_percent}% coverage, co-pay AED {self.__copay}")


class Patient(Person):
    """A patient with an account, personal details and medical information."""

    def __init__(self, patient_id, username, password, name, date_of_birth, phone,
                 email, address, emergency_contact, insurance=None, allergies="None",
                 medical_history="None"):
        super().__init__(patient_id, name, phone, email)
        self.__username = username
        self.__password = password
        self.__date_of_birth = date_of_birth
        self.__address = address
        self.__emergency_contact = emergency_contact
        self.__insurance = insurance          # Insurance object or None
        self.__allergies = allergies
        self.__medical_history = medical_history
        self.__appointments = []              # association with Appointment
        self.__records = []                   # visit records (EMR)
        self.__invoices = []

    # ----- account -----
    def get_username(self):
        return self.__username

    def check_password(self, password):
        return self.__password == password

    # ----- getters and setters -----
    def get_date_of_birth(self):
        return self.__date_of_birth

    def get_address(self):
        return self.__address

    def set_address(self, address):
        self.__address = address

    def get_emergency_contact(self):
        return self.__emergency_contact

    def set_emergency_contact(self, contact):
        self.__emergency_contact = contact

    def get_insurance(self):
        return self.__insurance

    def set_insurance(self, insurance):
        self.__insurance = insurance

    def get_allergies(self):
        return self.__allergies

    def set_allergies(self, allergies):
        self.__allergies = allergies

    def get_medical_history(self):
        return self.__medical_history

    def set_medical_history(self, history):
        self.__medical_history = history

    def get_appointments(self):
        return self.__appointments

    def get_records(self):
        return self.__records

    def get_invoices(self):
        return self.__invoices

    def add_record(self, record):
        self.__records.append(record)

    def add_invoice(self, invoice):
        self.__invoices.append(invoice)

    # ----- appointments -----
    def book_appointment(self, appointment):
        appointment.book(self)          # raises an error if the slot is taken
        self.__appointments.append(appointment)

    def cancel_appointment(self, appointment):
        appointment.cancel()
        self.__appointments.remove(appointment)

    def reschedule_appointment(self, old_appointment, new_appointment):
        """Move to a new slot: book the new one first, then cancel the old one."""
        if old_appointment not in self.__appointments:
            raise ValueError("You can only reschedule your own booked appointment.")
        self.book_appointment(new_appointment)
        self.cancel_appointment(old_appointment)

    def get_upcoming_appointments(self):
        return [a for a in self.__appointments if a.get_status() == "BOOKED"]

    # ----- visit summaries -----
    def download_visit_summaries(self, file_name):
        """Save all visit summaries to a text file."""
        with open(file_name, "w") as file:
            file.write(f"Royal Care Clinic - Visit summaries for {self._name}\n\n")
            for record in self.__records:
                file.write(record.get_summary(self) + "\n\n")
        return file_name

    def __str__(self):
        insurance = self.__insurance if self.__insurance else "No insurance"
        return (f"Patient: {self._name} (ID: {self._person_id})\n"
                f"  Date of birth     : {self.__date_of_birth}\n"
                f"  Phone / Email     : {self._phone} / {self._email}\n"
                f"  Address           : {self.__address}\n"
                f"  Emergency contact : {self.__emergency_contact}\n"
                f"  Insurance         : {insurance}\n"
                f"  Allergies         : {self.__allergies}\n"
                f"  Medical history   : {self.__medical_history}")


## 6. Invoice and Payments - `billing.py`

In [ ]:
%%writefile billing.py
"""
billing.py
Invoices and payments.

Invoice o-- Payment                    (aggregation)
Payment <|-- CardPayment               (inheritance)
Payment <|-- MobileWalletPayment       (inheritance)
Payment <|-- InsuranceClaim            (inheritance)
"""


class Payment:
    """Parent class for all payment methods."""

    def __init__(self, amount):
        self._amount = amount

    def get_amount(self):
        return self._amount

    def get_method(self):
        """Each child class returns its own payment method name."""
        return "Payment"

    def is_insurance(self):
        return False

    def __str__(self):
        return f"{self.get_method()}: AED {self._amount:.2f}"


class CardPayment(Payment):
    """Payment by debit or credit card."""

    def __init__(self, amount, card_last_four):
        super().__init__(amount)
        self.__card_last_four = card_last_four   # only the last 4 digits are kept

    def get_method(self):
        return f"Card (**** {self.__card_last_four})"


class MobileWalletPayment(Payment):
    """Payment by mobile wallet, e.g. Apple Pay."""

    def __init__(self, amount, wallet_name):
        super().__init__(amount)
        self.__wallet_name = wallet_name

    def get_method(self):
        return f"Mobile wallet ({self.__wallet_name})"


class InsuranceClaim(Payment):
    """The part of the bill paid by the insurance company."""

    def __init__(self, amount, insurance):
        super().__init__(amount)
        self.__insurance = insurance

    def get_method(self):
        return f"Insurance claim ({self.__insurance.get_provider()})"

    def is_insurance(self):
        return True


class Invoice:
    """The bill for one visit."""

    def __init__(self, invoice_id, record):
        self.__invoice_id = invoice_id
        self.__record = record               # the visit being billed
        self.__items = []                    # list of (description, amount)
        self.__payments = []                 # aggregation of Payment objects

        # Add all charges from the visit
        doctor = record.get_doctor()
        self.add_item(f"Consultation - {doctor.get_name()}", doctor.get_consultation_fee())
        for lab in record.get_lab_orders():
            self.add_item(f"Lab test - {lab.get_test_name()}", lab.get_price())
        for prescription in record.get_prescriptions():
            self.add_item(f"Medication - {prescription.get_drug_name()}",
                          prescription.get_price())

    def get_invoice_id(self):
        return self.__invoice_id

    def get_record(self):
        return self.__record

    def get_items(self):
        return self.__items

    def get_payments(self):
        return self.__payments

    def add_item(self, description, amount):
        """Add a charge, e.g. another service."""
        self.__items.append((description, amount))

    def get_total(self):
        return round(sum(amount for description, amount in self.__items), 2)

    def get_shares(self):
        """Return (insurance pays, patient pays)."""
        insurance = self.__record.get_patient().get_insurance()
        if insurance is None:
            return 0, self.get_total()
        return insurance.calculate_share(self.get_total())

    def get_amount_paid(self):
        return round(sum(p.get_amount() for p in self.__payments), 2)

    def get_balance(self):
        return round(self.get_total() - self.get_amount_paid(), 2)

    def get_status(self):
        if self.get_amount_paid() == 0:
            return "UNPAID"
        if self.get_balance() > 0:
            return "PARTIALLY PAID"
        return "PAID"

    def pay(self, payment):
        """Accept any kind of Payment (polymorphism)."""
        if payment.get_amount() <= 0 or payment.get_amount() > self.get_balance():
            raise ValueError("Invalid payment amount.")
        self.__payments.append(payment)

    def __str__(self):
        insurance_pays, patient_pays = self.get_shares()
        lines = ["=" * 54,
                 f"INVOICE {self.__invoice_id}  -  Royal Care Clinic",
                 f"Patient: {self.__record.get_patient().get_name()}",
                 "-" * 54]
        for description, amount in self.__items:
            lines.append(f"{description:<42}{amount:>12.2f}")
        lines += ["-" * 54,
                  f"{'Total':<42}{self.get_total():>12.2f}",
                  f"{'Insurance pays':<42}{insurance_pays:>12.2f}",
                  f"{'Patient pays':<42}{patient_pays:>12.2f}",
                  "-" * 54]
        for payment in self.__payments:
            lines.append(f"{'Paid by ' + payment.get_method():<42}{payment.get_amount():>12.2f}")
        lines += [f"{'Balance':<42}{self.get_balance():>12.2f}",
                  f"Status: {self.get_status()}",
                  "=" * 54]
        return "\n".join(lines)


## 7. Clinic and Department - `clinic.py`

In [ ]:
%%writefile clinic.py
"""
clinic.py
Clinic and Department classes.

Clinic *-- Department   (composition: departments are created by the clinic
                         and cannot exist without it)
Department o-- Staff    (aggregation: staff exist on their own and can move
                         to another department)
Clinic o-- Patient      (aggregation: the clinic keeps a list of registered patients)
"""

from patient import Patient
from staff import Doctor
from billing import Invoice


class Department:
    """A department of the clinic, e.g. Cardiology."""

    def __init__(self, name, operating_hours):
        self.__name = name
        self.__operating_hours = operating_hours
        self.__staff = []                  # aggregation

    def get_name(self):
        return self.__name

    def get_operating_hours(self):
        return self.__operating_hours

    def set_operating_hours(self, hours):
        self.__operating_hours = hours

    def get_staff(self):
        return self.__staff

    def add_staff(self, staff):
        """Add a staff member. If they were in another department, move them."""
        old_department = staff.get_department()
        if old_department is not None:
            old_department.remove_staff(staff)
        self.__staff.append(staff)
        staff.set_department(self)

    def remove_staff(self, staff):
        self.__staff.remove(staff)
        staff.set_department(None)

    def __str__(self):
        return f"{self.__name} ({self.__operating_hours}) - {len(self.__staff)} staff"


class Clinic:
    """The clinic: its information, departments, patients and invoices."""

    def __init__(self, name, address, phone, email, website):
        self.__name = name
        self.__address = address
        self.__phone = phone
        self.__email = email
        self.__website = website
        self.__departments = []     # composition
        self.__patients = []        # aggregation
        self.__invoices = []

    # ----- clinic information -----
    def get_name(self):
        return self.__name

    def get_address(self):
        return self.__address

    def set_address(self, address):
        self.__address = address

    def get_phone(self):
        return self.__phone

    def set_phone(self, phone):
        self.__phone = phone

    def get_email(self):
        return self.__email

    def set_email(self, email):
        self.__email = email

    def get_website(self):
        return self.__website

    def set_website(self, website):
        self.__website = website

    # ----- departments (composition) -----
    def add_department(self, name, operating_hours):
        """The clinic creates the department."""
        department = Department(name, operating_hours)
        self.__departments.append(department)
        return department

    def remove_department(self, name):
        """Close a department. Its staff are kept but have no department."""
        department = self.get_department(name)
        for staff in list(department.get_staff()):
            department.remove_staff(staff)
        self.__departments.remove(department)

    def get_department(self, name):
        for department in self.__departments:
            if department.get_name() == name:
                return department
        raise ValueError(f"No department called {name}.")

    def get_departments(self):
        return self.__departments

    # ----- staff and doctors -----
    def get_all_staff(self):
        staff_list = []
        for department in self.__departments:
            staff_list.extend(department.get_staff())
        return staff_list

    def get_doctors(self):
        return [s for s in self.get_all_staff() if isinstance(s, Doctor)]

    # ----- patients -----
    def register_patient(self, username, password, name, date_of_birth, phone, email,
                         address, emergency_contact, insurance=None, allergies="None",
                         medical_history="None"):
        """Create a patient account. Usernames must be unique."""
        for patient in self.__patients:
            if patient.get_username() == username:
                raise ValueError("This username is already taken.")
        patient_id = f"P{len(self.__patients) + 1:03d}"
        patient = Patient(patient_id, username, password, name, date_of_birth, phone,
                          email, address, emergency_contact, insurance, allergies,
                          medical_history)
        self.__patients.append(patient)
        return patient

    def login(self, username, password):
        for patient in self.__patients:
            if patient.get_username() == username and patient.check_password(password):
                return patient
        raise PermissionError("Wrong username or password.")

    def get_patients(self):
        return self.__patients

    # ----- appointment search -----
    def search_slots(self, date=None, doctor_name=None, speciality=None):
        """Find available slots by date, doctor name and/or speciality."""
        results = []
        for doctor in self.get_doctors():
            if doctor_name and doctor_name.lower() not in doctor.get_name().lower():
                continue
            if speciality and doctor.get_speciality() != speciality:
                continue
            results.extend(doctor.get_available_slots(date))
        return results

    # ----- billing -----
    def create_invoice(self, record):
        invoice = Invoice(f"INV-{len(self.__invoices) + 1:03d}", record)
        self.__invoices.append(invoice)
        record.get_patient().add_invoice(invoice)
        return invoice

    def get_invoices(self):
        return self.__invoices

    def __str__(self):
        lines = [f"{self.__name}",
                 f"  Address : {self.__address}",
                 f"  Phone   : {self.__phone}",
                 f"  Email   : {self.__email}",
                 f"  Website : {self.__website}",
                 "  Departments:"]
        for department in self.__departments:
            lines.append(f"    - {department}")
        return "\n".join(lines)


## 8. Sample data - `sample_data.py`

In [ ]:
%%writefile sample_data.py
"""
sample_data.py
Creates the Royal Care Clinic with its departments, staff and doctor
schedules. Used by main.py and by the test file.
"""

from clinic import Clinic
from staff import Staff, Doctor

NEXT_MONDAY = "2026-11-02"
NEXT_TUESDAY = "2026-11-03"


def create_clinic():
    """Build the clinic and return it together with the main staff objects."""
    clinic = Clinic("Royal Care Clinic", "Khalifa City, Abu Dhabi",
                    "+971 2 555 0100 / +971 50 555 0199",
                    "info@royalcare.ae", "www.royalcare.ae")

    # Departments (created by the clinic - composition)
    general = clinic.add_department("General Practice", "Sun-Thu 08:00-20:00")
    paediatrics = clinic.add_department("Paediatrics", "Mon-Fri 09:00-17:00")
    cardiology = clinic.add_department("Cardiology", "Mon-Wed 08:00-15:00")
    laboratory = clinic.add_department("Laboratory", "Daily 07:00-19:00")
    clinic.add_department("Radiology", "Mon-Fri 08:00-16:00")
    clinic.add_department("Pharmacy", "Daily 08:00-22:00")

    # Doctors
    dr_brown = Doctor("D001", "Dr. Brown", "050 111 2233", "brown@royalcare.ae",
                      "General Practitioner", "MBBS", "Mon-Fri 08:00-16:00", 300)
    dr_haddad = Doctor("D002", "Dr. Layla Haddad", "050 222 3344", "haddad@royalcare.ae",
                       "Paediatrician", "MD Paediatrics", "Mon-Thu 09:00-17:00", 350)
    dr_khalid = Doctor("D003", "Dr. Khalid", "050 333 4455", "khalid@royalcare.ae",
                       "Cardiologist", "MD Cardiology", "Mon-Wed 08:00-15:00", 500)

    # Other staff
    nurse = Staff("S001", "Fatima", "050 444 5566", "fatima@royalcare.ae",
                  "Nurse", "BSc Nursing", "Sun-Thu 08:00-16:00")
    lab_tech = Staff("S002", "Rahul", "050 555 6677", "rahul@royalcare.ae",
                     "Lab Technician", "BSc Medical Lab Science", "Sun-Thu 07:00-15:00")

    # Put staff in departments (aggregation)
    general.add_staff(dr_brown)
    general.add_staff(nurse)
    paediatrics.add_staff(dr_haddad)
    cardiology.add_staff(dr_khalid)
    laboratory.add_staff(lab_tech)

    # Doctor schedules: 30-minute slots
    for time in ["09:00", "09:30", "10:00", "10:30"]:
        dr_brown.add_slot(NEXT_MONDAY, time)
    for time in ["10:00", "10:30"]:
        dr_haddad.add_slot(NEXT_MONDAY, time)
    for time in ["09:00", "09:30", "10:00"]:
        dr_haddad.add_slot(NEXT_TUESDAY, time)
    dr_khalid.add_slot(NEXT_TUESDAY, "08:00")

    staff = {"brown": dr_brown, "haddad": dr_haddad, "khalid": dr_khalid,
             "nurse": nurse, "lab_tech": lab_tech}
    return clinic, staff


## 9. Main program - `main.py`

In [ ]:
%%writefile main.py
"""
main.py
Runs the Royal Care Clinic system with two example scenarios.
Run with:  python main.py
"""

from billing import CardPayment, MobileWalletPayment, InsuranceClaim
from patient import Insurance
from sample_data import create_clinic, NEXT_MONDAY


def title(text):
    """Print a section title."""
    print("\n" + "=" * 60)
    print(text)
    print("=" * 60)


def show_clinic(clinic, staff):
    title("CLINIC INFORMATION")
    print(clinic)
    clinic.set_website("www.royalcareclinic.ae")
    print(f"\nWebsite updated to: {clinic.get_website()}")

    print("\nStaff:")
    for person in clinic.get_all_staff():
        print(f"  {person}")


def scenario_aliyah(clinic, staff):
    title("SCENARIO 1: Aliyah visits Dr. Brown")
    dr_brown = staff["brown"]

    # 1. Create patient account
    insurance = Insurance("Daman", "DMN-7788", 80, 50)
    aliyah = clinic.register_patient("aliyah", "pass123", "Aliyah", "2001-03-14",
                                     "055 123 4567", "aliyah@email.com",
                                     "Al Reem Island, Abu Dhabi", "Imran (brother) 055 765 4321",
                                     insurance, "Penicillin", "Mild asthma")
    print("Account created:")
    print(aliyah)

    # 2. Search and book a 30-minute appointment next Monday
    print(f"\nGeneral Practitioner slots on {NEXT_MONDAY}:")
    slots = clinic.search_slots(date=NEXT_MONDAY, speciality="General Practitioner")
    for slot in slots:
        print(f"  {slot}")
    aliyah.book_appointment(slots[1])
    appointment = slots[1]
    print(f"\nBooked: {appointment}")

    # 3. Doctor records the visit, orders lab tests and prescribes
    record = dr_brown.record_visit(appointment, "Tonsillitis",
                                   "Sore throat and fever for 3 days",
                                   {"Temp": "38.4 C", "BP": "118/76", "HR": 96})
    blood_test = dr_brown.order_lab_test(record, "Complete Blood Count", 120)
    strep_test = dr_brown.order_lab_test(record, "Strep A Test", 90)
    prescription = dr_brown.prescribe(record, "Azithromycin", "500 mg",
                                      "Once a day for 3 days", 36)
    dr_brown.prescribe(record, "Paracetamol", "500 mg", "Every 6 hours if needed", 15)
    print(f"\nAppointment status : {appointment.get_status()}")
    print(f"Prescription status: {prescription.get_status()}")
    print(f"Lab test status    : {blood_test.get_status()}")

    # 4. Lab results are uploaded
    blood_test.add_result("WBC slightly high")
    strep_test.add_result("Positive")
    print(f"Lab test status after results: {blood_test.get_status()}")

    print("\n" + record.get_summary(aliyah))

    # 5. Invoice and payment: insurance claim + card
    invoice = clinic.create_invoice(record)
    insurance_pays, patient_pays = invoice.get_shares()
    invoice.pay(InsuranceClaim(insurance_pays, insurance))
    invoice.pay(CardPayment(patient_pays, "4417"))
    print("\n" + str(invoice))

    file_name = aliyah.download_visit_summaries("aliyah_visit_summaries.txt")
    print(f"\nVisit summaries downloaded to {file_name}")


def scenario_omar(clinic, staff):
    title("SCENARIO 2: Omar visits Dr. Haddad")
    dr_haddad = staff["haddad"]

    omar = clinic.register_patient("omar", "family2026", "Omar", "2019-06-02",
                                   "050 246 8101", "saeed.family@email.com",
                                   "MBZ City, Abu Dhabi", "Mariam (mother) 050 246 8101",
                                   allergies="Peanuts")
    print(f"Account created for {omar.get_name()} (no insurance)")

    # Search by doctor name, book, then reschedule
    slots = clinic.search_slots(doctor_name="Haddad")
    print(f"\nFree slots with Dr. Haddad: {len(slots)}")
    first = slots[0]
    omar.book_appointment(first)
    print(f"Booked      : {first}")
    new = slots[3]
    omar.reschedule_appointment(first, new)
    print(f"Rescheduled : {new}")
    print(f"Old slot is free again: {first.is_available()}")

    # Book a second appointment and cancel it
    extra = slots[4]
    omar.book_appointment(extra)
    omar.cancel_appointment(extra)
    print(f"Cancelled   : {extra}")
    print(f"Upcoming appointments for Omar: {len(omar.get_upcoming_appointments())}")

    # Visit with an attachment, prescription and lab test
    record = dr_haddad.record_visit(new, "Ear infection", "Pain in left ear",
                                    {"Temp": "38.9 C", "HR": 118})
    record.add_attachment("school_nurse_referral.pdf")
    dr_haddad.prescribe(record, "Amoxicillin syrup", "250 mg/5 ml", "7.5 ml twice a day", 38)
    hearing = dr_haddad.order_lab_test(record, "Ear pressure test", 150)
    hearing.add_result("Fluid behind left eardrum")
    print("\n" + record.get_summary(staff["nurse"]))

    # Access control: another patient cannot see Omar's record
    aliyah = clinic.login("aliyah", "pass123")
    try:
        record.get_summary(aliyah)
    except PermissionError as error:
        print(f"\nAccess denied: {error}")

    # Invoice with an extra service, paid by mobile wallet
    invoice = clinic.create_invoice(record)
    invoice.add_item("Ear examination", 60)
    invoice.pay(MobileWalletPayment(invoice.get_total(), "Apple Pay"))
    print("\n" + str(invoice))


def main():
    clinic, staff = create_clinic()
    show_clinic(clinic, staff)
    scenario_aliyah(clinic, staff)
    scenario_omar(clinic, staff)
    title("END")


if __name__ == "__main__":
    main()


## 10. Test cases - `test_clinic.py`

In [ ]:
%%writefile test_clinic.py
"""
test_clinic.py
Test cases for the Royal Care Clinic system (separate file).
Covers cases a-h from the assignment, the other features, and the two
example scenarios.

Run with:  python -m unittest -v test_clinic
"""

import os
import unittest

from billing import CardPayment, MobileWalletPayment, InsuranceClaim
from patient import Insurance
from sample_data import create_clinic, NEXT_MONDAY, NEXT_TUESDAY


class ClinicTest(unittest.TestCase):
    """setUp runs before every test, so each test starts with a fresh clinic."""

    def setUp(self):
        self.clinic, self.staff = create_clinic()
        self.dr_brown = self.staff["brown"]
        self.dr_haddad = self.staff["haddad"]
        self.insurance = Insurance("Daman", "DMN-7788", 80, 50)
        self.aliyah = self.clinic.register_patient(
            "aliyah", "pass123", "Aliyah", "2001-03-14", "055 123 4567",
            "aliyah@email.com", "Al Reem Island", "Imran 055 765 4321",
            self.insurance, "Penicillin", "Mild asthma")

    def book_and_visit(self):
        """Helper: Aliyah books Dr. Brown and the visit is recorded."""
        slot = self.dr_brown.get_available_slots(NEXT_MONDAY)[0]
        self.aliyah.book_appointment(slot)
        record = self.dr_brown.record_visit(slot, "Tonsillitis", "Sore throat",
                                            {"Temp": "38.4 C", "BP": "118/76"})
        return slot, record


# ---------------- Clinic information and staff ----------------
class TestClinicInformation(ClinicTest):

    def test_retrieve_clinic_info(self):
        self.assertEqual(self.clinic.get_name(), "Royal Care Clinic")
        self.assertEqual(self.clinic.get_email(), "info@royalcare.ae")
        self.assertEqual(len(self.clinic.get_departments()), 6)

    def test_update_clinic_info(self):
        self.clinic.set_phone("+971 2 555 0111")
        self.clinic.set_website("www.royalcareclinic.ae")
        self.assertEqual(self.clinic.get_phone(), "+971 2 555 0111")
        self.assertEqual(self.clinic.get_website(), "www.royalcareclinic.ae")

    def test_add_and_remove_department(self):
        self.clinic.add_department("Dermatology", "Mon-Thu 10:00-18:00")
        self.assertEqual(self.clinic.get_department("Dermatology").get_operating_hours(),
                         "Mon-Thu 10:00-18:00")
        self.clinic.remove_department("Dermatology")
        with self.assertRaises(ValueError):
            self.clinic.get_department("Dermatology")

    def test_staff_still_exist_when_department_closes(self):
        """Aggregation: removing a department does not delete its staff."""
        dr_khalid = self.staff["khalid"]
        self.clinic.remove_department("Cardiology")
        self.assertIsNone(dr_khalid.get_department())
        self.assertEqual(dr_khalid.get_name(), "Dr. Khalid")

    def test_staff_and_doctors(self):
        self.assertEqual(len(self.clinic.get_all_staff()), 5)
        self.assertEqual(len(self.clinic.get_doctors()), 3)
        self.assertEqual(self.dr_brown.get_role(), "Doctor")       # inherited from Staff
        self.assertEqual(self.dr_brown.get_department().get_name(), "General Practice")

    def test_move_staff_to_other_department(self):
        nurse = self.staff["nurse"]
        self.clinic.get_department("Paediatrics").add_staff(nurse)
        self.assertEqual(nurse.get_department().get_name(), "Paediatrics")
        self.assertNotIn(nurse, self.clinic.get_department("General Practice").get_staff())


# ---------------- a. Patient account creation ----------------
class TestPatientAccount(ClinicTest):

    def test_patient_details_stored(self):
        self.assertEqual(self.aliyah.get_name(), "Aliyah")
        self.assertEqual(self.aliyah.get_date_of_birth(), "2001-03-14")
        self.assertEqual(self.aliyah.get_address(), "Al Reem Island")
        self.assertEqual(self.aliyah.get_insurance().get_provider(), "Daman")
        self.assertEqual(self.aliyah.get_allergies(), "Penicillin")
        self.assertEqual(self.aliyah.get_medical_history(), "Mild asthma")
        self.assertIn(self.aliyah, self.clinic.get_patients())

    def test_login(self):
        self.assertIs(self.clinic.login("aliyah", "pass123"), self.aliyah)
        with self.assertRaises(PermissionError):
            self.clinic.login("aliyah", "wrong")

    def test_username_must_be_unique(self):
        with self.assertRaises(ValueError):
            self.clinic.register_patient("aliyah", "x", "Other", "2000-01-01", "050",
                                         "", "Dubai", "None")

    def test_update_profile(self):
        self.aliyah.set_phone("055 999 0001")
        self.aliyah.set_address("Saadiyat Island")
        self.assertEqual(self.aliyah.get_phone(), "055 999 0001")
        self.assertEqual(self.aliyah.get_address(), "Saadiyat Island")


# ---------------- b. Search for available appointments ----------------
class TestSearchAppointments(ClinicTest):

    def test_search_by_date(self):
        slots = self.clinic.search_slots(date=NEXT_MONDAY)
        self.assertEqual(len(slots), 6)        # Dr. Brown 4 + Dr. Haddad 2

    def test_search_by_doctor(self):
        slots = self.clinic.search_slots(doctor_name="Haddad")
        self.assertEqual(len(slots), 5)

    def test_search_by_speciality(self):
        slots = self.clinic.search_slots(speciality="Cardiologist")
        self.assertEqual(len(slots), 1)
        self.assertEqual(slots[0].get_doctor().get_name(), "Dr. Khalid")

    def test_booked_slot_not_listed(self):
        slot = self.dr_brown.get_available_slots()[0]
        self.aliyah.book_appointment(slot)
        self.assertNotIn(slot, self.clinic.search_slots(date=NEXT_MONDAY))


# ---------------- c. Make an appointment ----------------
class TestMakeAppointment(ClinicTest):

    def test_book_appointment(self):
        slot = self.dr_brown.get_available_slots(NEXT_MONDAY)[0]
        self.aliyah.book_appointment(slot)
        self.assertEqual(slot.get_status(), "BOOKED")
        self.assertIs(slot.get_patient(), self.aliyah)
        self.assertEqual(slot.get_duration(), 30)
        self.assertIn(slot, self.aliyah.get_upcoming_appointments())

    def test_cannot_double_book(self):
        slot = self.dr_brown.get_available_slots()[0]
        self.aliyah.book_appointment(slot)
        omar = self.clinic.register_patient("omar", "x", "Omar", "2019-06-02", "050",
                                            "", "MBZ City", "Mariam")
        with self.assertRaises(ValueError):
            omar.book_appointment(slot)

    def test_reschedule(self):
        old, new = self.dr_brown.get_available_slots()[0:2]
        self.aliyah.book_appointment(old)
        self.aliyah.reschedule_appointment(old, new)
        self.assertEqual(new.get_status(), "BOOKED")
        self.assertTrue(old.is_available())
        self.assertEqual(self.aliyah.get_appointments(), [new])
        other = self.dr_brown.get_available_slots()[0]
        with self.assertRaises(ValueError):          # old slot is no longer hers
            self.aliyah.reschedule_appointment(old, other)


# ---------------- d. Visit recording and EMR ----------------
class TestVisitRecording(ClinicTest):

    def test_record_linked_to_appointment(self):
        slot, record = self.book_and_visit()
        self.assertIs(slot.get_record(), record)
        self.assertIs(record.get_appointment(), slot)
        self.assertIs(record.get_patient(), self.aliyah)
        self.assertEqual(record.get_diagnosis(), "Tonsillitis")
        self.assertEqual(record.get_vitals()["BP"], "118/76")
        self.assertEqual(slot.get_status(), "COMPLETED")
        self.assertIn(record, self.aliyah.get_records())

    def test_attachment(self):
        slot, record = self.book_and_visit()
        record.add_attachment("xray_report.pdf")
        self.assertIn("xray_report.pdf", record.get_attachments())

    def test_wrong_doctor_cannot_record(self):
        slot = self.dr_brown.get_available_slots()[0]
        self.aliyah.book_appointment(slot)
        with self.assertRaises(PermissionError):
            self.dr_haddad.record_visit(slot, "Flu", "Notes", {})

    def test_access_control(self):
        slot, record = self.book_and_visit()
        self.assertIn("Tonsillitis", record.get_summary(self.aliyah))         # patient
        self.assertIn("Tonsillitis", record.get_summary(self.staff["nurse"]))  # staff
        other = self.clinic.register_patient("other", "x", "Other", "1990-01-01",
                                             "050", "", "Dubai", "None")
        with self.assertRaises(PermissionError):
            record.get_summary(other)

    def test_download_visit_summaries(self):
        self.book_and_visit()
        file_name = self.aliyah.download_visit_summaries("test_summary.txt")
        with open(file_name) as file:
            text = file.read()
        os.remove(file_name)
        self.assertIn("Tonsillitis", text)


# ---------------- e. Prescription generation ----------------
class TestPrescription(ClinicTest):

    def test_prescription_sent_to_pharmacy(self):
        slot, record = self.book_and_visit()
        prescription = self.dr_brown.prescribe(record, "Azithromycin", "500 mg",
                                               "Once a day for 3 days", 36)
        self.assertEqual(prescription.get_status(), "SENT TO PHARMACY")
        self.assertEqual(prescription.get_drug_name(), "Azithromycin")
        self.assertEqual(prescription.get_dosage(), "500 mg")
        self.assertEqual(prescription.get_directions(), "Once a day for 3 days")
        self.assertIn(prescription, record.get_prescriptions())


# ---------------- f. Lab order lifecycle ----------------
class TestLabOrder(ClinicTest):

    def test_lab_order_completed(self):
        slot, record = self.book_and_visit()
        lab = self.dr_brown.order_lab_test(record, "Blood Test", 120)
        self.assertEqual(lab.get_status(), "ORDERED")
        self.assertIsNone(lab.get_result())
        lab.add_result("Normal")
        self.assertEqual(lab.get_status(), "COMPLETED")
        self.assertEqual(lab.get_result(), "Normal")

    def test_result_linked_to_patient_and_visit(self):
        slot, record = self.book_and_visit()
        lab = self.dr_brown.order_lab_test(record, "Blood Test", 120)
        self.assertIn(lab, record.get_lab_orders())
        self.assertIs(record.get_patient(), self.aliyah)

    def test_result_cannot_be_added_twice(self):
        slot, record = self.book_and_visit()
        lab = self.dr_brown.order_lab_test(record, "Blood Test", 120)
        lab.add_result("Normal")
        with self.assertRaises(ValueError):
            lab.add_result("Changed")


# ---------------- g. Invoice generation and payment ----------------
class TestInvoice(ClinicTest):

    def make_invoice(self):
        slot, record = self.book_and_visit()
        self.dr_brown.order_lab_test(record, "Blood Test", 120)
        self.dr_brown.prescribe(record, "Azithromycin", "500 mg", "Daily", 36)
        return self.clinic.create_invoice(record)

    def test_invoice_total(self):
        invoice = self.make_invoice()
        self.assertEqual(invoice.get_total(), 456)        # 300 + 120 + 36
        invoice.add_item("Nebuliser", 44)
        self.assertEqual(invoice.get_total(), 500)

    def test_insurance_and_copay(self):
        invoice = self.make_invoice()
        # co-pay 50, insurance pays 80% of 406 = 324.80, patient pays 50 + 81.20
        self.assertEqual(invoice.get_shares(), (324.80, 131.20))

    def test_insurance_claim_and_card_payment(self):
        invoice = self.make_invoice()
        self.assertEqual(invoice.get_status(), "UNPAID")
        invoice.pay(InsuranceClaim(324.80, self.insurance))
        self.assertEqual(invoice.get_status(), "PARTIALLY PAID")
        invoice.pay(CardPayment(131.20, "4417"))
        self.assertEqual(invoice.get_status(), "PAID")
        self.assertEqual(invoice.get_balance(), 0)

    def test_mobile_wallet_and_overpayment(self):
        invoice = self.make_invoice()
        invoice.pay(MobileWalletPayment(400, "Apple Pay"))
        with self.assertRaises(ValueError):
            invoice.pay(MobileWalletPayment(100, "Apple Pay"))   # only 56 left

    def test_no_insurance(self):
        self.aliyah.set_insurance(None)
        invoice = self.make_invoice()
        self.assertEqual(invoice.get_shares(), (0, 456))


# ---------------- h. Appointment cancellation ----------------
class TestCancellation(ClinicTest):

    def test_cancel_appointment(self):
        slot = self.dr_brown.get_available_slots()[0]
        self.aliyah.book_appointment(slot)
        self.aliyah.cancel_appointment(slot)
        self.assertEqual(slot.get_status(), "CANCELLED")
        self.assertTrue(slot.is_available())                 # free for others
        self.assertEqual(self.aliyah.get_upcoming_appointments(), [])

    def test_cannot_cancel_completed_visit(self):
        slot, record = self.book_and_visit()
        with self.assertRaises(ValueError):
            slot.cancel()


# ---------------- Example scenario 1 ----------------
class TestScenarioAliyah(unittest.TestCase):

    def test_aliyah_visit(self):
        # 1. Create patient Aliyah and doctor Dr. Brown (General Practitioner)
        clinic, staff = create_clinic()
        dr_brown = staff["brown"]
        insurance = Insurance("Daman", "DMN-7788", 80, 50)
        aliyah = clinic.register_patient("aliyah", "pass123", "Aliyah", "2001-03-14",
                                         "055 123 4567", "aliyah@email.com",
                                         "Al Reem Island", "Imran", insurance)
        self.assertEqual(dr_brown.get_speciality(), "General Practitioner")

        # 2. Aliyah books a 30-minute appointment next Monday
        slot = clinic.search_slots(date=NEXT_MONDAY, speciality="General Practitioner")[0]
        aliyah.book_appointment(slot)
        self.assertEqual(slot.get_duration(), 30)

        # 3. Visit: diagnosis, lab tests, prescription
        record = dr_brown.record_visit(slot, "Tonsillitis", "Sore throat",
                                       {"Temp": "38.4 C"})
        blood = dr_brown.order_lab_test(record, "Complete Blood Count", 120)
        strep = dr_brown.order_lab_test(record, "Strep A Test", 90)
        prescription = dr_brown.prescribe(record, "Azithromycin", "500 mg", "Daily", 36)

        # 4. Lab results uploaded, invoice generated, payment completed
        blood.add_result("WBC slightly high")
        strep.add_result("Positive")
        invoice = clinic.create_invoice(record)
        insurance_pays, patient_pays = invoice.get_shares()
        invoice.pay(InsuranceClaim(insurance_pays, insurance))
        invoice.pay(CardPayment(patient_pays, "4417"))

        # 5. Assertions: visit data, prescription and billing are accurate
        self.assertEqual(record.get_diagnosis(), "Tonsillitis")
        self.assertEqual(slot.get_status(), "COMPLETED")
        self.assertEqual(blood.get_status(), "COMPLETED")
        self.assertEqual(prescription.get_status(), "SENT TO PHARMACY")
        self.assertEqual(invoice.get_total(), 546)            # 300 + 120 + 90 + 36
        self.assertEqual(invoice.get_shares(), (396.80, 149.20))
        self.assertEqual(invoice.get_status(), "PAID")


# ---------------- Example scenario 2 ----------------
class TestScenarioOmar(unittest.TestCase):

    def test_omar_visit(self):
        clinic, staff = create_clinic()
        dr_haddad = staff["haddad"]
        omar = clinic.register_patient("omar", "family2026", "Omar", "2019-06-02",
                                       "050 246 8101", "", "MBZ City", "Mariam")

        # Book, reschedule to Tuesday, then book and cancel another slot
        slots = clinic.search_slots(speciality="Paediatrician")
        omar.book_appointment(slots[0])
        omar.reschedule_appointment(slots[0], slots[3])
        omar.book_appointment(slots[4])
        omar.cancel_appointment(slots[4])
        self.assertEqual(omar.get_upcoming_appointments(), [slots[3]])
        self.assertEqual(slots[3].get_date(), NEXT_TUESDAY)

        # Visit with attachment, prescription and lab test
        record = dr_haddad.record_visit(slots[3], "Ear infection", "Left ear pain",
                                        {"Temp": "38.9 C"})
        record.add_attachment("referral.pdf")
        dr_haddad.prescribe(record, "Amoxicillin syrup", "250 mg/5 ml", "Twice a day", 38)
        lab = dr_haddad.order_lab_test(record, "Ear pressure test", 150)
        lab.add_result("Fluid behind eardrum")

        # No insurance: the family pays everything by mobile wallet
        invoice = clinic.create_invoice(record)
        invoice.add_item("Ear examination", 60)
        self.assertEqual(invoice.get_total(), 598)            # 350 + 150 + 38 + 60
        invoice.pay(MobileWalletPayment(598, "Apple Pay"))

        self.assertEqual(lab.get_status(), "COMPLETED")
        self.assertEqual(record.get_attachments(), ["referral.pdf"])
        self.assertEqual(invoice.get_status(), "PAID")


if __name__ == "__main__":
    unittest.main(verbosity=2)


## Run the program
`!python main.py` runs the main file, the same as typing it in a terminal.

In [ ]:
!python main.py

## Run the test cases
This runs every test in `test_clinic.py` and prints `ok` for each one that passes.

In [ ]:
!python -m unittest -v test_clinic